<div style="font-size: 0.85em;">
  <h3>Output Parsers – Key Concepts</h3>
  <ul>
    <li><strong>What are output parsers?</strong><br>
        Components that transform the raw LLM output (<code>AIMessage</code>) into a desired format such as a plain string, list, JSON, or Pydantic model.
    </li>
    <li><strong>Why use them?</strong>
      <ul>
        <li>Make LLM responses <strong>application‑ready</strong> (e.g., pass to a function, store in a database, display in a UI).</li>
        <li>Provide <strong>consistent and predictable</strong> output structures (e.g. a plain string, a list, or a table).</li>
        <li>Enable <strong>validation</strong> and, in advanced cases, <strong>automatic correction</strong> of malformed outputs.</li>
      </ul>
    </li>
    <li><strong>Where do they fit?</strong><br>
        Output parsers are added at the end of a chain using the pipe operator (<code>|</code>):<br>
        <code>Prompt → LLM → Output Parser → Final Output</code>
    </li>
    <li><strong>Common parser types:</strong>
      <table style="border-collapse: collapse; width: 100%;">
        <tr style="background-color: #5f1c1c;">
          <th style="border: 1px solid #ddd; padding: 4px;">Parser</th>
          <th style="border: 1px solid #ddd; padding: 4px;">Output Format</th>
          <th style="border: 1px solid #ddd; padding: 4px;">Use Case</th>
        </tr>
        <tr>
          <td style="border: 1px solid #ddd; padding: 4px;"><code>StrOutputParser</code></td>
          <td style="border: 1px solid #ddd; padding: 4px;">Plain string</td>
          <td style="border: 1px solid #ddd; padding: 4px;">Simple text answers</td>
        </tr>
        <tr>
          <td style="border: 1px solid #ddd; padding: 4px;"><code>CommaSeparatedListOutputParser</code></td>
          <td style="border: 1px solid #ddd; padding: 4px;">Python list</td>
          <td style="border: 1px solid #ddd; padding: 4px;">Lists of items, tags, categories</td>
        </tr>
        <tr>
          <td style="border: 1px solid #ddd; padding: 4px;"><code>JsonOutputParser</code></td>
          <td style="border: 1px solid #ddd; padding: 4px;">JSON object</td>
          <td style="border: 1px solid #ddd; padding: 4px;">Structured data with key‑value pairs</td>
        </tr>
        <tr>
          <td style="border: 1px solid #ddd; padding: 4px;"><code>PydanticOutputParser</code></td>
          <td style="border: 1px solid #ddd; padding: 4px;">Pydantic model</td>
          <td style="border: 1px solid #ddd; padding: 4px;">Validated, typed objects (covered later)</td>
        </tr>
      </table>
    </li>
    <li><strong>Example flow with <code>StrOutputParser</code>:</strong>
      <ol>
        <li>Prompt: <code>'Explain RAG in one sentence.'</code></li>
        <li>LLM returns: <code>AIMessage(content='RAG is ...')</code></li>
        <li>Parser returns: <code>'RAG is ...'</code> (a plain string)</li>
      </ol>
    </li>
    <li><strong>Key takeaway:</strong><br>
        Output parsers help you go from <em>raw model text</em> to <em>structured, usable data</em>, which is essential for building robust RAG pipelines and agents.
    </li>
  </ul>
</div>

<div style="font-size: 0.85em;">
  <h3>How a Chain Works (Output Parser at the End)</h3>
  <pre>
User Input (dict)
      |
      v
[Prompt Template]   -- fills placeholders, returns messages
      |
      v
[Chat Model]        -- returns AIMessage (contains content string)
      |
      v
[Output Parser]     -- transforms AIMessage into desired format
      |
      v
Final Output (string, list, JSON, Pydantic model)
  </pre>
</div>

Step 1: Imports and Environment

Import the two parsers to use in this lesson:
* StrOutputParser
* CommaSeparatedListOutputParser

In [1]:
from langchain_core.output_parsers import StrOutputParser, CommaSeparatedListOutputParser
from dotenv import load_dotenv

In [2]:
# Load environment variables from .env
load_dotenv()

True

Step 2: Build and Run a Chain with StrOutputParser

Create a chain that:
* Takes a question about RAG.
* Sends it to the model.
* Uses StrOutputParser to return just the answer text.

In [4]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

# Create a chat model
llm = ChatOpenAI(model='gpt-4o-mini', temperature=0)

# Create a prompt template
prompt = ChatPromptTemplate.from_messages([
    ('system', 'You are a helpful assistant.'),
    ('human', 'Explain {topic} in one sentence.')
])

# Build the chain with the output parser
chain = prompt | llm | StrOutputParser()

# Invoke the chain
response = chain.invoke({'topic': 'Machine learning'})
print('Result')
print(response)
print(f'\nType: {type(response)}')

Result
Machine learning is a subset of artificial intelligence that enables systems to learn from data, identify patterns, and make decisions with minimal human intervention.

Type: <class 'str'>


CommaSeparatedListOutputParser – What It Does
* This parser takes a string that contains comma-separated items and converts it into a Python list.
* Important: You must instruct the model in the prompt to output a comma-separated list. The parser then splits that string by commas and cleans up each item.

Example:
* Model output: 'python, fastapi, django, rag'
* Parser result: ['python', 'fastapi', 'django', 'rag']

Step 3: Build and Run a Chain with CommaSeparatedListOutputParser

In [10]:
prompt_lst = ChatPromptTemplate.from_messages([
    ('system', 'You are a helpful assistant.'),
    ('human', 'List of 4 languages in tech, separated by commas.')
])

# Build the chain with CommaSeparatedListOutputParser
chain_lst = prompt_lst | llm | CommaSeparatedListOutputParser()

# Invoke the chain
lst_response = chain_lst.invoke({})

print('Result:')
print(lst_response)
print(f'\nType: {type(lst_response)}')

Result:
['Python', 'JavaScript', 'Java', 'C++']

Type: <class 'list'>


<div style="font-size: 0.88em;">
  <h3>Real‑Life Uses of <code>CommaSeparatedListOutputParser</code></h3>
  <ul>
    <li><strong>Tag generation</strong> – Generate keywords/tags for a blog post or product.</li>
    <li><strong>Entity extraction</strong> – Extract names of people, companies, or locations from text.</li>
    <li><strong>Category classification</strong> – List categories that apply to a document or product.</li>
    <li><strong>Search query generation</strong> – Create multiple search queries for retrieval.</li>
    <li><strong>Data extraction for forms</strong> – Extract skills from a resume, etc.</li>
    <li><strong>User interest extraction</strong> – List interests from a conversation for recommendations.</li>
  </ul>
</div>